In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from PIL import Image
import cv2
import os

# --------------------- 多模态数据集类 ---------------------
class MultiModalDataset(Dataset):
    def __init__(self, image_paths, nir_data, labels, transform=None):
        self.image_paths = image_paths
        self.nir_data = nir_data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx][0]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        nir = torch.tensor(self.nir_data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, nir, label

# --------------------- 模型定义（需保持一致） ---------------------
class PyramidCNN(nn.Module):
    def __init__(self):
        super(PyramidCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x.view(x.size(0), -1)

class NIRAttentionExtractor(nn.Module):
    def __init__(self):
        super(NIRAttentionExtractor, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(64)
        )
        self.se_block = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.Sigmoid()
        )
        self.pre_gru = nn.Linear(128, 64)
        self.gru = nn.GRU(input_size=64, hidden_size=64, batch_first=True, bidirectional=True)
        self.attn_fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fusion = nn.Linear(192, 192)

    def forward(self, x):
        x_cnn = x.unsqueeze(1)
        x_cnn = self.conv(x_cnn)
        se = self.se_block(x_cnn).unsqueeze(-1)
        x_cnn = x_cnn * se
        x_cnn = x_cnn.mean(dim=-1)

        x_gru = self.pre_gru(x).unsqueeze(1)
        x_gru, _ = self.gru(x_gru)
        attn_weights = torch.softmax(self.attn_fc(x_gru), dim=1)
        x_gru = torch.sum(x_gru * attn_weights, dim=1)

        features = torch.cat([x_cnn, x_gru], dim=1)
        return self.fusion(features)

class MultiModalNet(nn.Module):
    def __init__(self):
        super(MultiModalNet, self).__init__()
        self.image_encoder = PyramidCNN()
        self.nir_encoder = NIRAttentionExtractor()
        self.fc = nn.Sequential(
            nn.Linear(320, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, img, nir):
        img_feat = self.image_encoder(img)
        nir_feat = self.nir_encoder(nir)
        fused = torch.cat([img_feat, nir_feat], dim=1)
        return self.fc(fused)

# --------------------- Grad-CAM类 ---------------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook()

    def hook(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(*input_tensor)
        if class_idx is None:
            class_idx = torch.argmax(output)
        output[:, class_idx].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-6)
        return cam

# --------------------- 加载数据与模型 ---------------------
data_root = r"L:\\常惠林\\萎凋\\所有样本"
nir_mat = sio.loadmat(r"L:\\常惠林\\萎凋\\NIR.mat")
nir_data = nir_mat['nir']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

from torchvision.datasets import ImageFolder
image_dataset = ImageFolder(root=data_root, transform=transform)
image_paths = image_dataset.imgs
labels = image_dataset.targets

# 随便选一个测试样本
index = 0
sample_dataset = MultiModalDataset(image_paths, nir_data, labels, transform=transform)
sample_img, sample_nir, _ = sample_dataset[index]
sample_img_input = sample_img.unsqueeze(0).cuda()
sample_nir_input = sample_nir.unsqueeze(0).cuda()

# 模型初始化与加载
model = MultiModalNet().cuda()
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# --------------------- Grad-CAM on RGB ---------------------
rgb_target_layer = model.image_encoder.layer3[0]
cam_rgb_generator = GradCAM(lambda img, nir: model(img, nir), rgb_target_layer)
cam_rgb = cam_rgb_generator.generate((sample_img_input, sample_nir_input))[0].cpu().numpy()

original_img = sample_img.permute(1, 2, 0).numpy()
heatmap_rgb = cv2.applyColorMap(np.uint8(255 * cam_rgb), cv2.COLORMAP_JET)
heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB)
overlay_rgb = 0.4 * heatmap_rgb / 255.0 + 0.6 * original_img
plt.imshow(np.clip(overlay_rgb, 0, 1))
plt.title("Grad-CAM on RGB")
plt.axis('off')
plt.show()

# --------------------- Grad-CAM on NIR ---------------------
nir_target_layer = model.nir_encoder.conv[3]
cam_nir_generator = GradCAM(lambda img, nir: model(img, nir), nir_target_layer)
cam_nir = cam_nir_generator.generate((sample_img_input, sample_nir_input))[0].cpu().numpy()

nir_img = sample_nir.numpy()
nir_img = (nir_img - nir_img.min()) / (nir_img.max() - nir_img.min())
nir_img = np.tile(nir_img, (3, 1))
nir_img = nir_img.reshape((8, 16, 3)).transpose(1, 0, 2)
nir_img = cv2.resize(nir_img, (224, 224))
heatmap_nir = cv2.applyColorMap(np.uint8(255 * cam_nir), cv2.COLORMAP_JET)
heatmap_nir = cv2.cvtColor(heatmap_nir, cv2.COLOR_BGR2RGB)
overlay_nir = 0.4 * heatmap_nir / 255.0 + 0.6 * nir_img
plt.imshow(np.clip(overlay_nir, 0, 1))
plt.title("Grad-CAM on NIR")
plt.axis('off')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'L:\\\\常惠林\\\\萎凋\\\\NIR.mat'